In [0]:
spark


# Connecting To ADLS Gen 2 Storage succesfull

In [0]:
storage_account = "your_storage_account_name"
application_id = "your_application_id"
directory_id = "your_directory_id"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", application_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", "your_secret_value")
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{directory_id}/oauth2/token")


# Reading The Data

In [0]:
# 0. Customers Dataset
customers_df=spark.read\
    .format("csv")\
    .option("header","true")\
     .option("inferSchema","true")\
    .load("abfss://olist-data@olistdatastoaccount.dfs.core.windows.net/bronze/olist_customers_dataset.csv")

# 1. Geolocation Dataset
geolocation_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://olist-data@olistdatastoaccount.dfs.core.windows.net/bronze/olist_geolocation_dataset.csv")

# 2. Order Items Dataset
items_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://olist-data@olistdatastoaccount.dfs.core.windows.net/bronze/olist_order_items_dataset.csv")

# 3. Order Payments Dataset
payments_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://olist-data@olistdatastoaccount.dfs.core.windows.net/bronze/olist_order_payments_dataset.csv")

# 4. Order Reviews Dataset
reviews_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://olist-data@olistdatastoaccount.dfs.core.windows.net/bronze/olist_order_reviews_dataset.csv")

# 5. Orders Dataset
orders_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://olist-data@olistdatastoaccount.dfs.core.windows.net/bronze/olist_orders_dataset.csv")

# 6. Products Dataset
products_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://olist-data@olistdatastoaccount.dfs.core.windows.net/bronze/olist_products_dataset.csv")

# 7. Sellers Dataset
sellers_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("abfss://olist-data@olistdatastoaccount.dfs.core.windows.net/bronze/olist_sellers_dataset.csv")




#Reading Data from PyMongo DB

In [0]:
from pymongo import MongoClient

In [0]:
# importing module
from pymongo import MongoClient

hostname = "your_host_name"
database = "your_database_name"
port = "your_port_number"
username = "your_username"
password = "your_password"

uri = "mongodb://" + username + ":" + password + "@" + hostname + ":" + port + "/" + database

# Connect with the portnumber and host
client = MongoClient(uri)

# Access database
mydatabase = client[database]


In [0]:
collection = mydatabase["product_categories"]

In [0]:
import pandas as pd
mongo_data=pd.DataFrame(list(collection.find()))

#Data Cleaning

In [0]:
from pyspark.sql.functions import *

In [0]:
def cleaneData(df,name):
    print("cleaning"+name)
    return df.dropDuplicates().na.drop("all")
orders_df=cleaneData(orders_df,"orders")
orders_df.show(10)    

cleaningorders
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|6f6665c1d76e55561...|12ae89712aa5a178c...|   delivered|     2017-02-23 08:03:00|2017-02-23 08:15:17|         2017-02-23 09:10:22|          2017-03-02 09:23:07|          2017-04-03 00:00:00|
|5c1f2bd7eff419c16...|ac4e328e156057d60...|   delivered|     2018-03-08 15:59:11|2018-03-08 16:31:43|         2018-03-13 01:21:57|          2018-03-18 18:26:36|          2018-03-20 00:00:00|
|38971812f1db9e4a3...|735774ca

In [0]:
# Convert DateColumns
orders_df = orders_df.withColumn("order_purchase_timestamp", to_date(col("order_purchase_timestamp")))\
    .withColumn("order_delivered_customer_date", to_date(col("order_delivered_customer_date")))\
    .withColumn("order_estimated_delivery_date", to_date(col("order_estimated_delivery_date")))

    

In [0]:
# Calculating Delivery & Time delays 
orders_df = orders_df.withColumn("actual_delivery_time", datediff("order_delivered_customer_date", "order_purchase_timestamp"))
orders_df = orders_df.withColumn("estimated_delivery_time", datediff("order_estimated_delivery_date", "order_purchase_timestamp"))
orders_df = orders_df.withColumn("Delay Time", col("actual_delivery_time") - col("estimated_delivery_time"))


In [0]:
orders_df.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-----------------------+----------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|actual_delivery_time|estimated_delivery_time|Delay Time|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-----------------------+----------+
|6f6665c1d76e55561...|12ae89712aa5a178c...|   delivered|              2017-02-23|2017-02-23 08:15:17|         2017-02-23 09:10:22|                   2017-03-02|                   2017-04-03|                   7|                     39|       -32|
|5c1f2bd7eff

# Joining DataSet

In [0]:
order_customer_df=orders_df.join(customers_df,"customer_id","left")
order_payment_df=order_customer_df.join(payments_df,"order_id","left")
order_item_df=order_payment_df.join(items_df,"order_id","left")
order_item_product_df=order_item_df.join(products_df,"product_id","left")
final_df=order_item_product_df.join(sellers_df,"seller_id","left")


# Mongo Data Enrichment

In [0]:
mongo_data.drop("_id",axis=1,inplace=True)
mongo_spark_df=spark.createDataFrame(mongo_data)
mongo_spark_df.show(5)

+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|         beleza_saude|                health_beauty|
| informatica_acess...|         computers_accesso...|
|           automotivo|                         auto|
|      cama_mesa_banho|               bed_bath_table|
|     moveis_decoracao|              furniture_decor|
+---------------------+-----------------------------+
only showing top 5 rows


In [0]:
final_df=final_df.join(mongo_spark_df,"product_category_name","left")

In [0]:
final_df.show(20)

+---------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-----------------------+----------+--------------------+------------------------+---------------+--------------+------------------+------------+--------------------+-------------+-------------+-------------------+------+-------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+----------------------+--------------------+------------+-----------------------------+
|product_category_name|           seller_id|          product_id|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|actual_delive

In [0]:
display(final_df)

product_category_name,seller_id,product_id,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,actual_delivery_time,estimated_delivery_time,Delay Time,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,payment_sequential,payment_type,payment_installments,payment_value,order_item_id,shipping_limit_date,price,freight_value,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,product_category_name_english
utilidades_domesticas,1b4c3a6f53068f0b6944d2d005c9fc89,e9a69340883a438c3f91739d14d3a56d,019886de8f385a39b75bedbb726fd4ef,8cf88d7ba142365ef2ca619ef06f9a0f,delivered,2018-02-10,2018-02-10T13:08:12Z,2018-02-14T15:28:51Z,2018-02-23,2018-03-14,13,32,-19,d29ede26cd3e2817b314005a88bd28a8,79092,campo grande,MS,1,credit_card,2,188.399994,1,2018-02-15T13:08:12Z,159.9,28.5,60,1912,5,3000,33,12,34,88730,sao ludgero,SC,housewares
telefonia,ea8482cd71df3c1969d7b9473ff13abc,036734b5a58d5d4f46b0616ddc047ced,01a6ad782455876aa89081449d49c452,71accffbcbdf8e02f67a469f65cdbf73,delivered,2018-01-18,2018-01-18T10:17:29Z,2018-01-22T22:37:04Z,2018-02-01,2018-02-20,14,33,-19,31c23262d79bc7e803884e37f9bc5359,98801,santo angelo,RS,1,credit_card,5,50.0900002,1,2018-01-24T10:17:29Z,34.99,15.1,58,751,5,300,17,4,12,4160,sao paulo,SP,telephony
cama_mesa_banho,d1c281d3ae149232351cd8c8cc885f0d,b1434a8f79cb3528540d9b21e686e823,01d907b3e209269e120a365fc2b97524,d02cc92f5e33eb58d9ff4d5cce6ae901,delivered,2017-08-09,2017-08-10T10:25:08Z,2017-08-11T19:05:53Z,2017-08-16,2017-08-29,7,20,-13,a7b781410bcc8bbf3221f48ff45aae6d,8503,ferraz de vasconcelos,SP,1,credit_card,10,169.759995,1,2017-08-16T10:25:08Z,151.99,17.77,57,184,1,13500,55,25,35,14940,ibitinga,SP,bed_bath_table
brinquedos,c8b3445d737de6befde0c88ede534a5e,d86a6c48f83b045cbba6df84926a1f25,028dc52e12ddda803ec1e35eb0b7b0d9,8c89a09d8fb33b6e5dc8a769d6b2bd63,delivered,2017-12-19,2017-12-19T10:53:22Z,2017-12-20T16:49:18Z,2017-12-21,2018-01-08,2,20,-18,148a35e3469010c9753897362d1eb8d0,8550,poa,SP,1,debit_card,1,61.7200012,1,2017-12-26T10:53:22Z,49.99,11.73,58,1150,2,3900,45,33,26,5734,sao paulo,SP,toys
relogios_presentes,6560211a19b47992c3666cc44a7e94c0,aa8d88eb4b9cb38894e33fa624c4287f,036dd381dfb3ec75e0a63e14828cc871,00f5116a953fdf1b86dd0deb055c0e12,delivered,2017-09-04,2017-09-04T22:43:55Z,2017-09-05T20:49:41Z,2017-09-13,2017-09-27,9,23,-14,9f268d26a5ab9839702b621fbcda4f5f,26325,queimados,RJ,1,credit_card,3,69.1399994,1,2017-09-11T22:43:55Z,55.0,14.14,54,335,4,250,16,2,11,5849,sao paulo,SP,watches_gifts
eletroportateis,9674754b5a0cb32b638cec001178f799,aa6746e94490239d3d9ee6ab89779aba,03ebfa9712b7dbc7031291856263b314,4024b83c510f004326fbcf0671738663,delivered,2018-03-23,2018-03-23T18:48:53Z,2018-03-26T21:17:58Z,2018-03-27,2018-04-05,4,13,-9,60389b9628af3e1ed0b3d14226b77994,9280,santo andre,SP,1,boleto,1,55.7799988,1,2018-03-29T18:48:53Z,46.9,8.88,37,90,1,500,16,35,25,4438,sao paulo,SP,small_appliances
brinquedos,406822777a0b9eb5c50e442dd4cd3ec5,5ca739ddd646d1ba53cca4e3c099a953,0420da8d50a3784011290a782f25a8a8,0949b5cf9adad08c1421aa3f1778e4a3,delivered,2018-08-09,2018-08-09T17:35:18Z,2018-08-13T12:48:00Z,2018-08-16,2018-08-21,7,12,-5,ab2731479851fc8019caa384772a2a0b,13563,sao carlos,SP,1,credit_card,1,74.6299973,1,2018-08-13T17:35:18Z,59.9,14.73,49,892,2,1200,52,13,37,18500,tatui,SP,toys
utilidades_domesticas,53e4c6e0f4312d4d2107a8c9cddf45cd,ac7e981115ad47f0e051f1b8b97e73b1,05afef1c185862cab9062b322ff25cc5,296de103322e463a1b76de3c81fdf02b,delivered,2017-07-25,2017-07-25T11:03:10Z,2017-07-31T15:49:35Z,2017-08-21,2017-09-04,27,41,-14,7857dff0702239bcf889f6af345fe607,59151,parnamirim,RN,1,credit_card,6,101.32,1,2017-07-31T11:03:10Z,27.99,22.67,42,241,2,500,30,30,30,13920,pedreira,SP,housewares
utilidades_domesticas,53e4c6e0f4312d4d2107a8c9cddf45cd,ac7e981115ad4

Databricks visualization. Run in Databricks to view.

In [0]:
final_df.write.mode("overwrite").parquet("abfss://olist-data@olistdatastoaccount.dfs.core.windows.net/silver")

In [0]:
# IF YOU FACE ANY ERROR REGARDING DUPLICATES , YOU CAN RUN THIS CODE AND GET RID OF DUPLICATES AND YOU CAN SAVE IT IN SILVER LAYER
def remove_duplicates(df):
    columns=df.columns
    seen_column=set()
    column_to_remove=[]
    for column in columns:
        if column in seen_column:
            column_to_remove.append(column)
        else:
            seen_column.add(column)
    df_cleaned=df.drop(*column_to_remove)
    return df_cleaned
final_df=remove_duplicates(final_df)
 